# Step 1.Notebook Purpose

# Step 2.Import Required Libraries

In [1]:
# Purpose  -->>  Import only what is required for training, evaluation, and saving the model.

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import joblib


In [2]:
import os
import sys

# Add src folder to Python path
sys.path.append(os.path.abspath("../src"))

# Import custom project functions
from preprocessing import clean_text
from feature_engineering import create_features

print("Custom functions imported successfully.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...


Custom functions imported successfully.


[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Step 3.Load Feature-Engineered Dataset

In [3]:
#Purpose  --> Load the dataset created in the feature engineering notebook. This dataset is now model-ready.

feature_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"
df = pd.read_csv(feature_path)
df.head()


,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx balance,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.0,0.0


# Step 4.Separate Features and Target

In [4]:
# Purpose -->> Clearly separate: X → input features, y → target variable (risk or sentiment label)

In [5]:
df.columns

Index(['char_length', 'word_length', 'stopword_ratio', 'risk_keyword_flag',
       'product_encoded', 'issue_encoded', 'complaint_year', 'complaint_month',
       'accordance', 'account',
       ...
       'xxxx balance', 'xxxx date', 'xxxx xxxx', 'xxxx xxxxxxxx', 'xxxxxxxx',
       'xxxxxxxx balance', 'xxxxxxxx xxxx', 'xxxxyear', 'year', 'yet'],
      dtype='object', length=308)

In [6]:
#Purpose -- Combine multiple signals into a single numeric score.

#We will use: risk_keyword_flag → already created (0 or 1),and word_length → long complaints often indicate severity

In [7]:
df['risk_score'] = (
    df['risk_keyword_flag'] +
    (df['word_length'] > 100).astype(int)
)


In [8]:
# Convert Risk Score → Target Label

# Purpose : Machine learning models need categorical targets, not scores.

In [9]:
def map_risk(score):
    if score == 0:
        return 0   # Low
    elif score == 1:
        return 1   # Medium
    else:
        return 2   # High

df['risk_label'] = df['risk_score'].apply(map_risk)


In [10]:
# Validate the Target Variable

# Purpose : Ensure the label exists and is balanced.

In [11]:
df['risk_label'].value_counts()

risk_label
2    1448
1    1195
0     681
Name: count, dtype: int64

In [12]:
# This confirms: 1.Target created, 2.Classification problem is valid

In [13]:
df.shape

(3324, 310)

In [14]:
#Check Column Presence 

#Purpose : Avoid KeyError later.

In [15]:
'risk_label' in df.columns


True

In [16]:
#Remove Helper Column (Optional but Professional)

#Purpose : risk_score is only for label creation, not for training.

In [17]:
df.drop(columns=['risk_score'], inplace=True)


In [18]:
#Save Updated Dataset

#Purpose : Persist the dataset with features + target.

In [19]:
feature_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\data\processed\complaints_features.csv"
df.to_csv(feature_path, index=False)

print("Dataset with target variable saved successfully")


Dataset with target variable saved successfully


In [20]:
#Confirm Final Shape

#Purpose : Final sanity check before model training.

In [21]:
df.shape


(3324, 309)

In [22]:
X = df.drop(columns=['risk_label'])
y = df['risk_label']

In [23]:
X

,char_length,word_length,stopword_ratio,risk_keyword_flag,product_encoded,issue_encoded,complaint_year,complaint_month,accordance,account,...,xxxx balance,xxxx date,xxxx xxxx,xxxx xxxxxxxx,xxxxxxxx,xxxxxxxx balance,xxxxxxxx xxxx,xxxxyear,year,yet
0,57,8,0.0,0,6,32,2025,10,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
1,82,14,0.0,1,6,32,2025,10,0.555462,0.200713,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
2,26,5,0.0,0,7,32,2020,5,0.000000,1.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3,364,54,0.0,1,6,32,2025,10,0.000000,0.169315,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
4,1599,233,0.0,1,6,32,2025,11,0.000000,0.112413,...,0.0,0.0,0.409834,0.040575,0.032309,0.0,0.039342,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3319,339,49,0.0,0,6,32,2025,11,0.000000,0.000000,...,0.0,0.0,0.117449,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3320,1513,279,0.0,1,10,77,2025,9,0.000000,0.121353,...,0.0,0.0,0.235962,0.000000,0.041854,0.0,0.000000,0.0,0.000000,0.064745
3321,257,39,0.0,1,8,4,2025,9,0.000000,0.253319,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000
3322,737,107,0.0,0,6,32,2025,11,0.245237,0.088615,...,0.0,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000


In [24]:
y

0       0
1       1
2       0
3       1
4       2
       ..
3319    0
3320    2
3321    1
3322    1
3323    1
Name: risk_label, Length: 3324, dtype: int64

In [25]:
print(X.shape)
print(y.value_counts())

(3324, 308)
risk_label
2    1448
1    1195
0     681
Name: count, dtype: int64


# Model Training

In [28]:
# Purpose : To evaluate the model on unseen data (real-world simulation).

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [ ]:
# Save feature names used during training
feature_columns = X_train.columns.tolist()

print("Total feature columns:", len(feature_columns))

In [ ]:
# Purpose : Create the final model using production-safe settings.

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    n_jobs=-1
)


In [ ]:
# Purpose : Learn patterns from the training data.

model.fit(X_train, y_train)

In [ ]:
print(model.classes_)

In [ ]:
# Purpose : Test how well the model generalizes.

y_pred = model.predict(X_test)

In [ ]:
y_pred

In [ ]:
sample_text = "Multiple unauthorized transactions detected and bank failed to respond"

cleaned = clean_text(sample_text)

features = create_features(
    text=cleaned,
    product="Credit Card",
    issue="Fraud"
)

import pandas as pd

df_sample = pd.DataFrame([features])
df_sample = df_sample.reindex(columns=feature_columns, fill_value=0)

print("Prediction:", model.predict(df_sample))
print("Probabilities:", model.predict_proba(df_sample))
print("Classes:", model.classes_)

In [ ]:
# Purpose : Measure quality using business-relevant metrics.

from sklearn.metrics import classification_report, f1_score

print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

In [ ]:
# Purpose : Understand false positives & false negatives.

from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

In [ ]:
# I use confusion matrix to analyze misclassification impact in risk detection.

In [ ]:
# Purpose : Use the model later in deployment (FastAPI).

import joblib

model_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\models\logistic_regression_model.pkl"
joblib.dump(model, model_path)

print("Model saved successfully")


In [ ]:
# Purpose : Ensure feature alignment during inference.

feature_cols_path = r"C:\Users\hp\Desktop\Fintech_Complaint_Analysis\models\feature_columns.pkl"
joblib.dump(X.columns.tolist(), feature_cols_path)

print("Feature columns saved")

In [ ]:
print("Model training pipeline completed successfully")